In [ ]:
import os, glob

cands = sorted(set(
    glob.glob('/kaggle/input/*knee*') +
    glob.glob('/kaggle/input/*/*knee*')
))
print("Candidatos:", cands)
DATA = cands[0]
print("DATA =", DATA)

for name in sorted(os.listdir(DATA)):
    p = os.path.join(DATA, name)
    if os.path.isdir(p):
        print(f"[DIR ] {name:40s} {len(os.listdir(p))} entradas")
    else:
        print(f"[FILE] {name:40s} {os.path.getsize(p)/1e6:.2f} MB")

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None, 'display.width', 200, 'display.max_colwidth', 80)

csvs = sorted(glob.glob(os.path.join(DATA, '*.csv')))
dfs = {}
for f in csvs:
    df = pd.read_csv(f)
    dfs[os.path.basename(f)] = df
    print("=" * 100)
    print(os.path.basename(f), df.shape)
    print(df.dtypes)
    display(df.head(3))
    print("Nulos por columna:\n", df.isna().sum())


In [ ]:
LABELS = ['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA','Lateral OA',
          'PF OA','Effusion','Synovitis',"Baker's",'Contusion','Fracture']
train = dfs['train.csv']
series = dfs['train_series.csv']

gold = train.dropna(subset=LABELS, how='all').copy()
print("Estudios dorados:", len(gold))
print("¿Alguno dorado con NaN parcial?:", gold[LABELS].isna().any(axis=1).sum())
print("Valores únicos en etiquetas:", sorted(pd.unique(gold[LABELS].values.ravel())))

prev = pd.DataFrame({'positivos': gold[LABELS].sum().astype(int),
                     'prevalencia_%': (gold[LABELS].mean()*100).round(1)})
display(prev.sort_values('positivos'))

In [ ]:
!pip install -q langdetect
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0

def lang(t):
    try: return detect(str(t)[:1000])
    except Exception: return 'unk'

train['lang'] = train['Report'].apply(lang)
train['is_gold'] = train['StudyInstanceUID'].isin(gold['StudyInstanceUID'])
train['report_len'] = train['Report'].str.len()

tab = pd.crosstab(train['lang'], train['is_gold'], margins=True)
tab.columns = ['no_dorado', 'dorado', 'total']
display(tab.sort_values('total', ascending=False))
print(train['report_len'].describe())

In [ ]:
series['combo'] = (series['Anatomical_Plane'] + ' | fluid=' + series['Fluid_Sensitive'].astype(str)
                   + ' | FS=' + series['Fat_Suppression'].astype(str))
print("Series por estudio:\n", series.groupby('StudyInstanceUID').size().describe())
display(series['combo'].value_counts().to_frame('n_series'))

# ¿Cuántos estudios tienen cada combinación? (lo que importa para elegir la secuencia)
cov = series.groupby('combo')['StudyInstanceUID'].nunique().sort_values(ascending=False)
display((cov.to_frame('n_estudios').assign(pct=lambda d: (d.n_estudios/train.shape[0]*100).round(1))))

# Cobertura de las candidatas en los 58 dorados
g = series[series['StudyInstanceUID'].isin(gold['StudyInstanceUID'])]
display(g.groupby('combo')['StudyInstanceUID'].nunique().sort_values(ascending=False).to_frame('dorados_con_esta_serie'))

In [ ]:
SAG = (series.Anatomical_Plane=='Sagittal') & (series.Fluid_Sensitive==1) & (series.Fat_Suppression==1)
COR = (series.Anatomical_Plane=='Coronal')  & (series.Fluid_Sensitive==1) & (series.Fat_Suppression==1)

gold_ids = set(gold.StudyInstanceUID)
sin_sag = gold_ids - set(series[SAG].StudyInstanceUID)
sin_cor = gold_ids - set(series[COR].StudyInstanceUID)
print("Dorados sin sagital FS:", len(sin_sag)); display(series[series.StudyInstanceUID.isin(sin_sag)][['StudyInstanceUID','combo']])
print("Dorados sin coronal FS:", len(sin_cor)); display(series[series.StudyInstanceUID.isin(sin_cor)][['StudyInstanceUID','combo']])

dup = series[SAG].groupby('StudyInstanceUID').size()
print("Estudios con >1 sagital FS:", (dup>1).sum(), "| distribución:", dup.value_counts().to_dict())

In [ ]:
!pip install -q pydicom
import pydicom, numpy as np

TS = os.path.join(DATA, 'train_series')
s0 = series[SAG].iloc[0]
sdir = os.path.join(TS, s0.StudyInstanceUID, s0.SeriesInstanceUID)
print("Existe la ruta estudio/serie:", os.path.isdir(sdir))
print("Contenido del estudio:", os.listdir(os.path.join(TS, s0.StudyInstanceUID))[:10])
files = sorted(os.listdir(sdir))
print("Nº de archivos en la serie:", len(files), "| ejemplo:", files[:3])

ds = pydicom.dcmread(os.path.join(sdir, files[0]))
for tag in ['Modality','Manufacturer','ManufacturerModelName','MagneticFieldStrength','SeriesDescription',
            'ProtocolName','Rows','Columns','PixelSpacing','SliceThickness','SpacingBetweenSlices',
            'PhotometricInterpretation','BitsStored','InstanceNumber','ImagePositionPatient',
            'ImageOrientationPatient','InstitutionName','Laterality','BodyPartExamined']:
    print(f"{tag:28s}", getattr(ds, tag, '—'))
arr = ds.pixel_array
print("pixel_array:", arr.shape, arr.dtype, "min/max:", arr.min(), arr.max(),
      "p1/p99:", np.percentile(arr, [1, 99]))

In [ ]:
import matplotlib.pyplot as plt

def load_series(study, ser):
    d = os.path.join(TS, study, ser)
    dss = [pydicom.dcmread(os.path.join(d, f)) for f in os.listdir(d)]
    dss.sort(key=lambda x: int(getattr(x, 'InstanceNumber', 0)))
    return np.stack([x.pixel_array.astype(np.float32) for x in dss])

def norm(img):
    lo, hi = np.percentile(img, [1, 99]); return np.clip((img-lo)/(hi-lo+1e-6), 0, 1)

study = list(set(series[SAG].StudyInstanceUID) & set(series[COR].StudyInstanceUID) & gold_ids)[0]
sub = series[(series.StudyInstanceUID==study) & (series.Fluid_Sensitive==1)]
fig, axes = plt.subplots(1, len(sub), figsize=(5*len(sub), 5))
for ax, (_, r) in zip(np.atleast_1d(axes), sub.iterrows()):
    vol = load_series(study, r.SeriesInstanceUID)
    ax.imshow(norm(vol[len(vol)//2]), cmap='gray'); ax.axis('off')
    ax.set_title(f"{r.Anatomical_Plane} | {vol.shape}")
plt.tight_layout(); plt.show()
print(gold[gold.StudyInstanceUID==study][LABELS].T)

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

def info(row):
    d = os.path.join(TS, row.StudyInstanceUID, row.SeriesInstanceUID)
    out = {'StudyInstanceUID': row.StudyInstanceUID, 'SeriesInstanceUID': row.SeriesInstanceUID}
    try:
        fs = [f for f in os.listdir(d) if f.endswith('.dcm')]
        out['n_slices'] = len(fs)
        ds = pydicom.dcmread(os.path.join(d, fs[0]), stop_before_pixels=True)
        ps = getattr(ds, 'PixelSpacing', [None, None])
        out.update(Rows=getattr(ds,'Rows',None), Columns=getattr(ds,'Columns',None),
                   PixelSpacing=float(ps[0]) if ps[0] else None,
                   SliceThickness=getattr(ds,'SliceThickness',None),
                   SpacingBetweenSlices=getattr(ds,'SpacingBetweenSlices',None),
                   Manufacturer=str(getattr(ds,'Manufacturer','')),
                   Model=str(getattr(ds,'ManufacturerModelName','')),
                   FieldStrength=getattr(ds,'MagneticFieldStrength',None),
                   SeriesDescription=str(getattr(ds,'SeriesDescription','')))
    except Exception as e:
        out['error'] = str(e)
    return out

rows = list(series.itertuples(index=False))
with ThreadPoolExecutor(16) as ex:
    res = list(tqdm(ex.map(info, rows), total=len(rows)))

cat = series.merge(pd.DataFrame(res), on=['StudyInstanceUID','SeriesInstanceUID'])
cat['is_gold'] = cat.StudyInstanceUID.isin(gold_ids)
cat = cat.merge(train[['StudyInstanceUID','lang']], on='StudyInstanceUID', how='left')
cat.to_csv('/kaggle/working/series_catalog.csv', index=False)
train[['StudyInstanceUID','lang','is_gold','report_len']].to_csv('/kaggle/working/study_lang.csv', index=False)
print(cat.shape, "| errores:", cat['error'].notna().sum() if 'error' in cat else 0)

In [ ]:
num = ['n_slices','Rows','PixelSpacing','SliceThickness']
display(cat.groupby('combo')[num].describe().T.round(2))

cat['vendor'] = cat.Manufacturer.str.split().str[0].str.upper()
st = cat.drop_duplicates('StudyInstanceUID')
display(st.vendor.value_counts().to_frame('estudios'))
display(pd.crosstab(st.lang, st.vendor))          # proxy de centro × idioma
display(st.FieldStrength.value_counts().to_frame('estudios'))